In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("../../Data/top100_ai_tools_2026.csv")
df

,Rank,Tool_Name,Primary_Category,Sub_Category,Pricing_Model,Starting_Price_USD,Monthly_Traffic_Est,User_Rating,Agentic_Capability_Score,API_Available,Release_Year,Active_Users_Est,Description
0,1,ChatGPT,Conversational AI,General Assistant,Freemium,20.00,1800000000,4.80,9.50,Yes,2022,180000000,"OpenAI's flagship conversational AI for text, ..."
1,2,Gemini,Conversational AI,Multimodal AI,Freemium,19.99,500000000,4.70,9.30,Yes,2023,90000000,Google's multimodal AI integrated with Google ...
2,3,Canva AI,Image Generation,Design Assistant,Freemium,15.00,500000000,4.60,6.00,Yes,2023,170000000,AI design tools integrated in Canva
3,4,Character.AI,Conversational AI,Roleplay & Chat,Freemium,9.99,300000000,4.30,6.00,No,2022,20000000,AI characters for entertainment and roleplay
4,5,Claude,Conversational AI,General Assistant,Freemium,20.00,200000000,4.80,9.40,Yes,2023,50000000,Anthropic's safety-focused AI assistant with l...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,496,Koral Bot,Research & Search,Market Research,Paid,196.25,114463,4.10,6.08,No,2023,38201,AI research assistant for faster knowledge dis...
496,497,Quill Flow,HR & Recruiting,Employee Onboarding,Paid,71.45,101182,4.18,7.39,Yes,2023,3464,AI recruitment and HR automation platform
497,498,Cipher GPT,Legal AI,Legal Research,Paid,414.84,88779,4.38,7.20,Yes,2023,18600,"AI platform for legal research, drafting, and ..."
498,499,Quill GPT,HR & Recruiting,Employee Onboarding,Paid,84.36,80270,4.22,7.35,Yes,2021,14974,AI recruitment and HR automation platform


In [3]:
df['worth it'] = np.random.randint(0,2, df.shape[0])


In [4]:
def add_nan_random(df, columns, percent=0.05):
    df_copy = df.copy()

    # subset only selected columns
    sub_df = df_copy[columns]

    total_cells = sub_df.size
    n_nan = int(total_cells * percent)

    for _ in range(n_nan):
        i = np.random.randint(0, sub_df.shape[0])
        j = np.random.randint(0, sub_df.shape[1])
        sub_df.iat[i, j] = np.nan

    # put modified columns back
    df_copy[columns] = sub_df

    return df_copy

# Apply function
df = add_nan_random(df,['Pricing_Model', 'Starting_Price_USD', 'Monthly_Traffic_Est',], 0.05)

In [5]:
df.isnull().mean() * 100

Rank                        0.0
Tool_Name                   0.0
Primary_Category            0.0
Sub_Category                0.0
Pricing_Model               5.2
Starting_Price_USD          4.8
Monthly_Traffic_Est         5.0
User_Rating                 0.0
Agentic_Capability_Score    0.0
API_Available               0.0
Release_Year                0.0
Active_Users_Est            0.0
Description                 0.0
worth it                    0.0
dtype: float64

In [6]:
df.columns

Index(['Rank', 'Tool_Name', 'Primary_Category', 'Sub_Category',
       'Pricing_Model', 'Starting_Price_USD', 'Monthly_Traffic_Est',
       'User_Rating', 'Agentic_Capability_Score', 'API_Available',
       'Release_Year', 'Active_Users_Est', 'Description', 'worth it'],
      dtype='str')

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['worth it']), df['worth it'], test_size=0.2, random_state=42)

In [8]:
df['Pricing_Model'].value_counts()

Pricing_Model
Paid           237
Freemium       199
Free            23
Pay-per-use     14
API/Paid         1
Name: count, dtype: int64

In [9]:
num = ['Starting_Price_USD', 'Monthly_Traffic_Est', 'User_Rating', 'Agentic_Capability_Score', 'Active_Users_Est']

cat = ['Primary_Category','Pricing_Model', 'API_Available', 'Release_Year']

In [10]:
from sklearn.preprocessing import StandardScaler,MinMaxScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [11]:
from sklearn.pipeline import Pipeline
impute_scale_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("scale", StandardScaler())
])

In [12]:
impute_ordinal_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(categories=[['Free', 'Freemium', 'Pay-per-use', 'Paid', 'API/Paid']]))
])

In [20]:
transformer = ColumnTransformer(transformers=[
    ("trf1", impute_scale_pipe,['Starting_Price_USD', 'Monthly_Traffic_Est','User_Rating']),
    ("trf2", StandardScaler(),['Agentic_Capability_Score', 'Active_Users_Est','Release_Year']),
    ("trf3", OneHotEncoder(sparse_output=False, drop='first',dtype=np.int32),["Primary_Category"]),
    ("trf4", impute_ordinal_pipe ,['Pricing_Model']),
    ("trf5", OrdinalEncoder(),['API_Available'])
]
)

# ,remainder="passthrough"

In [21]:
X_train_trf = transformer.fit_transform(X_train)
X_test_trf = transformer.transform(X_test)

In [22]:
X_train_trf

array([[-0.20823205, -0.18871694, -0.80193428, ...,  0.        ,
         1.        ,  1.        ],
       [-0.1311724 , -0.21492392, -0.97629941, ...,  0.        ,
         3.        ,  1.        ],
       [-0.54269392,  1.0933619 ,  1.29044725, ...,  0.        ,
         1.        ,  1.        ],
       ...,
       [ 0.40813011, -0.20209466, -0.04635206, ...,  0.        ,
         1.        ,  0.        ],
       [-0.49588306, -0.21496814, -0.04635206, ...,  0.        ,
         3.        ,  1.        ],
       [-0.64045615, -0.06170464,  0.12801306, ...,  0.        ,
         1.        ,  1.        ]], shape=(400, 29))

In [23]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_encod = le.fit_transform(y_train)
y_test_encod = le.transform(y_test)

# Testing

In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import  accuracy_score
lr = LogisticRegression()
lr.fit(X_train_trf,y_train_encod)
y_pred = lr.predict(X_test_trf)
accuracy_score(y_test_encod, y_pred)

0.5